# Round 3 — Data overview

Loads `prices_round_3_day_*.csv` (semicolon-separated), lists products, basic book stats, and groups names by role (spot vs VEV ladder). See `DATA_CONNECTIONS.txt` in the parent `ROUND3` folder for how files relate to each other.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


def _find_r3() -> Path:
    """Data may live in ROUND3/ or ROUND3/round3/; notebook cwd may be ROUND3 or notebooks/."""
    for base in (Path("."), Path(".."), Path("../..")):
        p = base.resolve()
        if (p / "prices_round_3_day_0.csv").is_file() or (p / "round3" / "prices_round_3_day_0.csv").is_file():
            return p
    raise FileNotFoundError("Could not find prices_round_3_day_0.csv — run Jupyter from ROUND3/ or ROUND3/notebooks/.")


R3 = _find_r3()


def price_csv(day: int) -> Path:
    for p in (R3 / f"prices_round_3_day_{day}.csv", R3 / "round3" / f"prices_round_3_day_{day}.csv"):
        if p.is_file():
            return p
    raise FileNotFoundError(f"No prices CSV for day {day} under {R3}")


def trades_csv(day: int) -> Path | None:
    for p in (R3 / f"trades_round_3_day_{day}.csv", R3 / "round3" / f"trades_round_3_day_{day}.csv"):
        if p.is_file():
            return p
    return None


DAY = 0
print("ROUND3 data root:", R3)
path = price_csv(DAY)
print("Using:", path)
df = pd.read_csv(path, sep=";")
print("rows:", len(df), "  columns:", list(df.columns))

In [ ]:
prods = sorted(df["product"].unique())
print(len(prods), "products:", prods)
ts = df["timestamp"]
print(
    "timestamps:", int(ts.min()), "…", int(ts.max()),
    " step:", int(ts.diff().mode().iloc[0]) if len(ts) > 1 else "n/a",
)
rows_per = df.groupby("product").size()
assert (rows_per == rows_per.iloc[0]).all(), "uneven row counts per product"
print("rows per product:", int(rows_per.iloc[0]))

In [ ]:
d = df.copy()
d["spread"] = d["ask_price_1"] - d["bid_price_1"]
sp = d.groupby("product").agg(
    mean_mid=("mid_price", "mean"),
    mean_spread=("spread", "mean"),
    med_spread=("spread", "median"),
)
sp = sp.sort_values("mean_spread", ascending=False)
sp.style.format({"mean_mid": "{:.2f}", "mean_spread": "{:.2f}", "med_spread": "{:.0f}"})

In [ ]:
vev = [p for p in prods if p.startswith("VEV_")]
vev_order = sorted(vev, key=lambda s: int(s.split("_", 1)[1]))
groups = {
    "spot / cash-like": [p for p in prods if p in ("VELVETFRUIT_EXTRACT", "HYDROGEL_PACK")],
    "VEV ladder (strike index)": vev_order,
}
for name, members in groups.items():
    print(name + ":", members)

In [ ]:
tpath = trades_csv(DAY)
if tpath is None:
    print("No trades file found for this day.")
else:
    trades = pd.read_csv(tpath, sep=";")
    print("Trades file:", tpath.name, "  rows:", len(trades))
    print(trades.head())
    print("\nTrades per symbol (top 15):\n", trades["symbol"].value_counts().head(15))